# Passing Behavior with Lambdas

CSC-239 · Module 10 · Lesson 1 of 4

You can define interfaces, pass object references to methods, and call an implementation through an interface type. This lesson uses that foundation to pass a small processing rule as a value.

Select the **Java** kernel in your Workspace. Start with a fresh kernel and run cells in order. This notebook creates its own starting state.


## Learning Goals

- Assign and pass a lambda through a functional interface.
- Use a compatible method reference and explain the restriction on captured local values.


## Why This Matters

A help desk prints labels for visitors and rooms. The steps for receiving text and producing a label stay the same, but the text rule changes. Supplying the rule lets one method serve several needs.


## Check Your Starting Point

Recall what an interface promises and how a method can accept an interface-typed parameter. Explain the difference between assigning a reference and invoking a method through it. Retrieve the meaning of the two type arguments in a generic type.

**My explanation:**


## Concept

### Describe one operation with a functional interface

A **functional interface** has one abstract method describing an operation. It may also have default methods or static methods; those do not add another required abstract operation. This lesson uses Java's existing Function interface instead of writing a new interface.

Import Function from java.util.function. In `Function<String, String>`, the first type argument is the input type and the second is the result type. The operation is apply: it accepts the input and returns the result. Both types happen to be String here, but they have different jobs.

### Supply the behavior with a lambda

A **lambda expression** supplies behavior for a compatible functional interface. It has parameters to the left of -> and a body to the right. The compiler uses the receiving interface type to check the parameter and result types.

```java
import java.util.function.Function;
Function<String, String> emphasize = text -> text + "!";
System.out.println(emphasize.apply("Ready"));
System.out.println(emphasize.apply("Go"));
```

This prints Ready! and Go! on separate lines. The interface type tells Java that text is a String. The body produces a String by concatenating an exclamation mark. For this single-expression form, the expression's value is the returned result; no return statement is written.

The assignment supplies a rule. It does not process every possible input at that moment. Each apply call supplies one input and invokes the rule. Creating a lambda and invoking its operation are separate actions.

The parameter text belongs to the lambda's operation. It is not the name of an input variable that every caller must declare. Use a descriptive parameter name that helps explain the rule.

### Pass the rule to another method

**Behavior as an argument** means passing a functional-interface value so the receiving method can invoke the supplied operation. Java still passes an argument value by copying it; the method receives a reference to the supplied behavior.

```java
import java.util.function.Function;
class TextRules {
    static String render(String input, Function<String, String> rule) {
        return rule.apply(input);
    }
}
Function<String, String> welcome = name -> "Welcome, " + name;
System.out.println(TextRules.render("Ari", welcome));
System.out.println(TextRules.render("Bo", name -> "Hello, " + name));
```

This prints Welcome, Ari and Hello, Bo. render knows the operation's input and result types. It does not need to know which greeting the supplied rule will use. The second call passes a lambda directly where the functional-interface argument is expected.

Compare this with the interfaces you implemented in Module 6. A lambda is useful for a small operation that matches one abstract-method contract. It does not mean every interface can be replaced with a lambda. An interface with two unrelated required abstract operations needs a different implementation approach.

### Use an enclosing local value

A lambda can also use a value from the surrounding method. A **captured local value** is an enclosing local value used by the lambda. Such a local variable or parameter must be final or effectively final. The Java **final** modifier prevents reassignment after a variable receives its initial value. **Effectively final** means a local variable or parameter is assigned its value and is not reassigned afterward, even without the final modifier.

```java
import java.util.function.Function;
class PrefixRules {
    static Function<String, String> withPrefix(String prefix) {
        return text -> prefix + text;
    }
}
Function<String, String> guest = PrefixRules.withPrefix("Guest: ");
Function<String, String> staff = PrefixRules.withPrefix("Staff: ");
System.out.println(guest.apply("Maya"));
System.out.println(staff.apply("Luis"));
```

This prints Guest: Maya and Staff: Luis. Each withPrefix call receives a parameter value and supplies a rule using that value. The parameter is never reassigned. The returned rule remains usable after withPrefix returns.

The following replacement method is intentionally invalid and is for reading only:

```text
static Function<String, String> withPrefix(String prefix) {
    prefix = prefix.trim();
    return text -> prefix + text;
}
```

Reassigning prefix makes that parameter ineligible for capture. One repair is to assign the cleaned text to a new local variable and never reassign that variable. Another is to clean the argument before calling the original helper. Do not paste this invalid method into an ordinary runnable cell.

In IJava, top-level notebook variables are handled differently from local variables inside an ordinary method. Use the complete named-method example to investigate this local-variable rule. A successful experiment with a top-level notebook variable does not disprove the rule for a method's local variables or parameters.

Capturing a reference does not make its object immutable. This lesson captures String values and uses no changing shared object as part of the text rule.

### Refer to an existing method

A **method reference** supplies compatible behavior by referring to an existing method. String::trim can be used where `Function<String, String>` is required. Here the incoming String becomes the object whose trim method is called.

```java
import java.util.function.Function;
Function<String, String> clean = String::trim;
System.out.println(clean.apply("  lab  "));
```

This prints lab. In this context, String::trim supplies the same operation as text -> text.trim(). The double colon names the method as behavior to use later; it does not immediately call trim. The apply call supplies the input String and starts the operation.

Use the form that makes the rule clearest. A method reference works when an existing method already matches the required operation. A lambda is useful when the rule needs an expression such as adding a label around the input.


## Video Demonstration

Watch one render method apply different supplied rules. Identify which call provides the text and which call invokes the chosen operation.

<video controls preload="metadata" width="960">
  <source src="media/01_passing_behavior_with_lambdas/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/01_passing_behavior_with_lambdas/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the passing behavior with lambdas demonstration transcript](media/01_passing_behavior_with_lambdas/transcript.md).


## Worked Example

**Subgoal 1: state the operation type.** Give render a `Function<String, String>` parameter.

**Subgoal 2: supply two rules.** Use one lambda to trim text and another to add a guest label.

**Subgoal 3: reuse an existing operation.** Supply String::trim through the same interface type.


In [ ]:
import java.util.function.Function;
class LabelPrinter {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = text -> text.trim();
Function<String, String> tag = text -> "Guest: " + text;
System.out.println(LabelPrinter.render("  Maya  ", clean));
System.out.println(LabelPrinter.render("Luis", tag));
Function<String, String> namedClean = String::trim;
System.out.println(LabelPrinter.render("  Nora  ", namedClean));


Expected output:

```text
Maya
Guest: Luis
Nora
```

render invokes apply with its text argument. The first rule removes surrounding spaces, the second adds a prefix, and the method reference supplies the existing trim behavior. Each invocation produces one String result.


## Predict, Run, Trace, and Explain

### Predict three applications of text rules

Before running, predict the three printed lines. For each call, identify its input String, the supplied rule and the operation that produces the result. Explain whether assigning `clean`, `tag` or `namedClean` prints anything or invokes a text operation by itself.

My three predicted lines:

Input, supplied rule and returned value for each call:

What assigning each functional value does:


In [ ]:
import java.util.function.Function;
class LabelPrinter {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = text -> text.trim();
Function<String, String> tag = text -> "Guest: " + text;
System.out.println(LabelPrinter.render("  Iris  ", clean));
System.out.println(LabelPrinter.render("Owen", tag));
Function<String, String> namedClean = String::trim;
System.out.println(LabelPrinter.render("  Bea  ", namedClean));


Run the complete prediction program in the Workspace. Compare all three lines with your prediction. Keep the original prediction and write a post-run explanation of any correction. Identify the call that applies each rule; distinguish the String input from the functional value passed beside it.

My original prediction:

Actual three lines:

What I confirmed or corrected:

Where each supplied operation is invoked:

### Trace the input, behavior and result

Trace the second `LabelPrinter.render` call into the helper and back to the print statement. Record the two argument values, the helper parameters receiving them, the input to `apply` and the returned String. Then explain why the first and third calls both remove surrounding spaces even though their rule declarations use different syntax. What single abstract operation makes `Function<String, String>` suitable for these rules?

Arguments and receiving parameters for the tag call:

Input to apply and returned result:

Why the lambda and method reference both trim:

Function’s required abstract operation:

<details>
<summary>Show answer</summary>

The first call passes the padded Iris String and the clean value to render. render calls `rule.apply(text)`, so the clean lambda returns Iris without surrounding spaces. The second call applies the tag lambda to Owen and returns Guest: Owen. The last call applies the `String::trim` method reference to the padded Bea String and returns Bea. These are the three printed lines, in order. Assigning a lambda or method reference supplies behavior; the render calls invoke it through apply. In `Function<String, String>`, the first String is the input type and the second is the result type. In the second call, the helper receives Owen as text and the tag functional value as rule. Calling apply invokes the supplied operation and returns its String to render, which returns it to the print statement. The clean lambda and method reference provide the same trim operation in this context; neither declaration trims every possible String in advance. Function has one abstract operation, apply.

```java
import java.util.function.Function;
class LabelPrinter {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = text -> text.trim();
Function<String, String> tag = text -> "Guest: " + text;
System.out.println(LabelPrinter.render("  Iris  ", clean));
System.out.println(LabelPrinter.render("Owen", tag));
Function<String, String> namedClean = String::trim;
System.out.println(LabelPrinter.render("  Bea  ", namedClean));
```

Expected output:

```text
Iris
Guest: Owen
Bea
```

Common error: Treating assignment of a rule as a call that processes text. Assuming every supplied rule trims the input. Reading the second type argument as a second input.

</details>


### Keep a prefix available to a returned rule

Predict all three printed lines before running this complete program. Mark the two calls to `withPrefix` and the later two calls to `apply`. For each returned rule, identify the surrounding method parameter it uses and the input supplied later. Explain why `prefix` is effectively final in this method: it receives its value and is not reassigned. Does the method have to still be running when the returned rule is applied? Keep your prediction and explain the actual results after running.

My three predicted lines:

Prefix associated with each returned rule:

Where the rules are created and where they are applied:

Actual output and my post-run explanation:

Why the local capture is permitted:


In [ ]:
import java.util.function.Function;
class PrefixMaker {
    static Function<String, String> withPrefix(String prefix) {
        return text -> prefix + text;
    }
}
Function<String, String> first = PrefixMaker.withPrefix("Desk: ");
Function<String, String> second = PrefixMaker.withPrefix("Room: ");
System.out.println("Rules ready");
System.out.println(first.apply("East"));
System.out.println(second.apply("West"));


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

The two withPrefix calls create separate functional values using the parameter values Desk: and Room:, each followed by a space. Rules ready is printed before either apply call. first.apply("East") returns Desk: East; second.apply("West") returns Room: West. Each returned rule remains usable after the helper returns. The method parameter prefix is effectively final because the method never reassigns it. This is a local parameter inside a named method, so it is the correct place to examine the local capture rule. The later text input belongs to each apply call; it is separate from the earlier prefix value.

```java
import java.util.function.Function;
class PrefixMaker {
    static Function<String, String> withPrefix(String prefix) {
        return text -> prefix + text;
    }
}
Function<String, String> first = PrefixMaker.withPrefix("Desk: ");
Function<String, String> second = PrefixMaker.withPrefix("Room: ");
System.out.println("Rules ready");
System.out.println(first.apply("East"));
System.out.println(second.apply("West"));
```

Expected output:

```text
Rules ready
Desk: East
Room: West
```

Common error: Thinking the later text input is supplied when withPrefix is called. Assuming the second helper call replaces the prefix used by the first returned rule. Using a top-level notebook variable to draw conclusions about a method-local capture.

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete a method that receives a rule

Complete the displayed program by replacing `FUNCTION_TYPE`, `INVOKE` and `ARROW`. Use `Function<String, String>` for the helper’s rule parameter, the interface operation for its call, and the lambda arrow between the parameter and body. Copy the completed program into the empty work cell and run it. It should print `Note: Map`. Explain why the helper receives both a String and a functional value and why the rule returns a String.

This sample is for repair:

```java
import java.util.function.Function;
class NotePrinter {
    static String render(String text, FUNCTION_TYPE rule) {
        return rule.INVOKE(text);
    }
}
Function<String, String> notice = text ARROW "Note: " + text;
System.out.println(NotePrinter.render("Map", notice));
```


My three substitutions:

Actual output:

Why the two helper arguments have different jobs:

<details>
<summary>Show answer</summary>

Use `Function<String, String>` for FUNCTION_TYPE, apply for INVOKE and -> for ARROW. The helper accepts a String input and a compatible functional value. Its apply call runs the notice operation with Map, producing Note: Map. render returns that String; the caller prints it. The full program includes the import, helper, rule and caller, so it does not depend on an earlier cell.

```java
import java.util.function.Function;
class NotePrinter {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> notice = text -> "Note: " + text;
System.out.println(NotePrinter.render("Map", notice));
```

Expected output:

```text
Note: Map
```

Common error: Omitting the Function import in the completed program. Calling the functional value with ordinary method-call syntax instead of apply. Leaving a placeholder in executable code.

</details>


### Change the supplied labeling rule

The starter is the complete prediction program. Change only the prefix in the tag lambda from `Guest: ` to `Visitor: `. Keep the trailing space inside the prefix. Predict which output line changes and why the other two keep their behavior, then run your edit. Next change only the tag call’s input from `Owen` to `Kai` and test again. Explain why `LabelPrinter.render` requires no change for either check.


In [ ]:
import java.util.function.Function;
class LabelPrinter {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = text -> text.trim();
Function<String, String> tag = text -> "Guest: " + text;
System.out.println(LabelPrinter.render("  Iris  ", clean));
System.out.println(LabelPrinter.render("Owen", tag));
Function<String, String> namedClean = String::trim;
System.out.println(LabelPrinter.render("  Bea  ", namedClean));


Predicted line affected by the rule change:

Actual output after changing the prefix:

Actual output after changing the caller input:

Why the helper stayed unchanged:

<details>
<summary>Show answer</summary>

The tag value now supplies the expression "Visitor: " + text. Applied to Owen, it returns Visitor: Owen. The first and third rules still trim their own inputs, so their outputs remain Iris and Bea. The same helper continues to invoke whichever functional value it receives. Changing the second caller input to Kai changes only that rule’s returned label, giving Visitor: Kai. A caller value and the operation applied to it are separate choices.

```java
import java.util.function.Function;
class LabelPrinter {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = text -> text.trim();
Function<String, String> tag = text -> "Visitor: " + text;
System.out.println(LabelPrinter.render("  Iris  ", clean));
System.out.println(LabelPrinter.render("Owen", tag));
Function<String, String> namedClean = String::trim;
System.out.println(LabelPrinter.render("  Bea  ", namedClean));
```

Expected output:

```text
Iris
Visitor: Owen
Bea
```

Common error: Changing the helper to print one hard-coded visitor name. Changing clean or namedClean even though only tag’s prefix needs modification. Dropping the space after the colon.

**Additional test: `Modified tag called with Kai`.** The same Visitor rule now receives Kai. The other two inputs and rules are unchanged, so Iris and Bea remain the first and third lines.

```java
import java.util.function.Function;
class LabelPrinter {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = text -> text.trim();
Function<String, String> tag = text -> "Visitor: " + text;
System.out.println(LabelPrinter.render("  Iris  ", clean));
System.out.println(LabelPrinter.render("Kai", tag));
Function<String, String> namedClean = String::trim;
System.out.println(LabelPrinter.render("  Bea  ", namedClean));
```

Expected output:

```text
Iris
Visitor: Kai
Bea
```

</details>


### Repair a reassigned captured parameter

The displayed program is intentionally invalid. Read it before editing. In `withPrefix`, identify the reassigned parameter that the lambda tries to use. Explain why that parameter is not effectively final. Repair the method by storing the formatted prefix in a different local String that is not reassigned, and let the lambda use that local value. Keep the helper calls and print statements unchanged. Put only your complete repaired program in the work cell and run it. Then change the first helper argument from `Help` to `Office` and explain the new output. Investigate the local rule inside this named method; top-level IJava variables are handled differently.

This sample is for repair:

```java
import java.util.function.Function;
class PrefixRepair {
    static Function<String, String> withPrefix(String prefix) {
        prefix = "[" + prefix + "] ";
        return text -> prefix + text;
    }
}
Function<String, String> first = PrefixRepair.withPrefix("Help");
Function<String, String> second = PrefixRepair.withPrefix("Lab");
System.out.println(first.apply("Mina"));
System.out.println(second.apply("Kai"));
```


The reassigned local parameter and the violated rule:

My repair and why its local value qualifies:

Actual repaired output:

Actual Office test and what it checks:

<details>
<summary>Show answer</summary>

The faulty method assigns prefix a new formatted value before the lambda captures it. That reassignment prevents the method parameter from being effectively final. The repair initializes a new local String, labelPrefix, from the original parameter and never reassigns labelPrefix. The lambda therefore may use it. The first helper call returns a rule using [Help] followed by a space, and the second returns a rule using [Lab] followed by a space. Applying them produces [Help] Mina and [Lab] Kai. A captured local reference must meet the local-variable rule; moving the experiment to top-level notebook variables would test a different situation.

```java
import java.util.function.Function;
class PrefixRepair {
    static Function<String, String> withPrefix(String prefix) {
        String labelPrefix = "[" + prefix + "] ";
        return text -> labelPrefix + text;
    }
}
Function<String, String> first = PrefixRepair.withPrefix("Help");
Function<String, String> second = PrefixRepair.withPrefix("Lab");
System.out.println(first.apply("Mina"));
System.out.println(second.apply("Kai"));
```

Expected output:

```text
[Help] Mina
[Lab] Kai
```

Common error: Keeping the reassignment to prefix and only changing the lambda’s parameter name. Reassigning labelPrefix after initializing it. Replacing each rule with a fixed complete result, so later input no longer matters.

**Additional test: `Repaired helper receives Office instead of Help`.** Only the first captured prefix changes. The later Mina and Kai inputs and the second Lab prefix remain unchanged.

```java
import java.util.function.Function;
class PrefixRepair {
    static Function<String, String> withPrefix(String prefix) {
        String labelPrefix = "[" + prefix + "] ";
        return text -> labelPrefix + text;
    }
}
Function<String, String> first = PrefixRepair.withPrefix("Office");
Function<String, String> second = PrefixRepair.withPrefix("Lab");
System.out.println(first.apply("Mina"));
System.out.println(second.apply("Kai"));
```

Expected output:

```text
[Office] Mina
[Lab] Kai
```

</details>


## Independent Practice

### Supply separate cleaning and bracketing rules

Define `RoomLabels.render(String text, Function<String, String> rule)` to invoke the supplied rule and return its result. Create separate values for `String::trim` and a lambda that surrounds its input with square brackets. Clean the supplied room name `"  lab  "`, then bracket the cleaned result; also bracket empty text directly. Print `[lab]` and `[]` on separate lines. Include the import, class, rules and caller setup in your own complete program. Explain which operation runs when each functional value is applied.

My complete program:

Actual baseline output:

Input, functional value and result for each application:

Why the helper can serve both operations:


### Check different and empty room text

Test your complete room-label program with `"  studio  "`, whitespace-only text `"   "`, and already-clean `"hall"` in place of `"  lab  "`. Keep the separate direct empty-text bracket call in every test. Before each run, predict both lines and the intermediate cleaned String. After running, compare exact brackets and spaces and explain the results. Restore and rerun the original lab case. Explain why whitespace-only input and the direct empty input can reach the same bracketed result through different sequences of calls.

For each input, my predicted cleaned String and two lines:

Actual output for studio, whitespace-only and hall:

Why whitespace-only cleaning and direct empty bracketing agree:

Restored baseline output and explanation:


<details>
<summary>Show answer</summary>

`RoomLabels.render` returns `rule.apply(text)`, so the supplied functional value determines the operation. The clean value uses `String::trim`; applied to the padded lab text, it returns lab. The bracket lambda then receives that cleaned String and returns [lab]. The separate call passes empty text directly to bracket and returns []. Cleaning and bracketing happen at their respective applications, in that order. The program includes its Function import, helper, both rules and all caller setup. The boundary cases separate a changed room, removal of all surrounding whitespace, and an input that needs no trimming. Each still applies the clean rule before the bracket rule for the room, then applies bracket directly to empty text. Restoring the original caller checks that the baseline still prints [lab] and [].

```java
import java.util.function.Function;
class RoomLabels {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = String::trim;
Function<String, String> bracket = text -> "[" + text + "]";
String room = RoomLabels.render("  lab  ", clean);
System.out.println(RoomLabels.render(room, bracket));
System.out.println(RoomLabels.render("", bracket));
```

Expected output:

```text
[lab]
[]
```

Common error: Bracketing the padded room before cleaning, leaving spaces inside the brackets. Hard-coding lab inside render instead of invoking its rule parameter. Trying to call String::trim as though it were an already computed String.

**Additional test: Different padded room: studio.** trim returns studio; bracket then returns [studio]. The independent direct empty-text application still returns [].

```java
import java.util.function.Function;
class RoomLabels {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = String::trim;
Function<String, String> bracket = text -> "[" + text + "]";
String room = RoomLabels.render("  studio  ", clean);
System.out.println(RoomLabels.render(room, bracket));
System.out.println(RoomLabels.render("", bracket));
```

Expected output:

```text
[studio]
[]
```

**Additional test: Whitespace-only room text.** The first line results from cleaning spaces to an empty String and then bracketing it. The second brackets an already empty String directly. Both produce [], but the first uses two applications and the second uses one.

```java
import java.util.function.Function;
class RoomLabels {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = String::trim;
Function<String, String> bracket = text -> "[" + text + "]";
String room = RoomLabels.render("   ", clean);
System.out.println(RoomLabels.render(room, bracket));
System.out.println(RoomLabels.render("", bracket));
```

Expected output:

```text
[]
[]
```

**Additional test: Already-clean room: hall.** trim leaves the text hall unchanged. The following bracket application returns [hall]; the separate empty application returns [].

```java
import java.util.function.Function;
class RoomLabels {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = String::trim;
Function<String, String> bracket = text -> "[" + text + "]";
String room = RoomLabels.render("hall", clean);
System.out.println(RoomLabels.render(room, bracket));
System.out.println(RoomLabels.render("", bracket));
```

Expected output:

```text
[hall]
[]
```

</details>


## Summary

A functional interface describes one required abstract operation. A lambda supplies compatible behavior, and apply invokes a Function with an input. Passing a functional value lets a method use a supplied rule. Captured local variables and parameters must be final or effectively final. A method reference names an existing compatible operation.

Close the answers and distinguish creating a rule, passing it, and invoking it.


## Reflection

Choose a text-processing need from your field. Describe two rules that could share the same input and result types. Explain what the receiving method would know and which detail each supplied rule would decide.

**My design and explanation:**

Next, you will apply these small operations to collection elements through a stream pipeline.


## Supplemental Reading

- [Java functional interfaces](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/function/package-summary.html) describes standard interfaces for supplied operations.
- [Function and its apply operation](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/function/Function.html) defines the input and result type parameters.
- [Lambda bodies and captured local values](https://docs.oracle.com/javase/specs/jls/se21/html/jls-15.html#jls-15.27.2) specifies the local-variable capture rule.
